# FinQA RAG: Build embeddings on GPU (Google Colab)

**Why:** Embedding the full FinQA corpus on CPU is slow (e.g. 6+ minutes for ~50k chunks). Running on Colab with a **free T4 GPU** is much faster (typically 1–2 minutes with batch_size=256).

**Steps:**
1. **Enable GPU:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.
2. Run the cells below. They clone the repo (or use your fork), install deps, build the index on GPU, and create a zip to download.
3. Download `finqa_retriever_index.zip`, unzip it, and place the folder **`finqa_retriever_index`** at  
   `data/rag/FinQA/train/finqa_retriever_index/` in your local repo.
4. Run `python eval_runner.py ... --category rag --dataset FinQA` locally; it will **load the pre-built index** and skip embedding.

**Is Colab best for free-tier GPU?** For this use case, yes: free T4 (15GB), no credit card for basic GPU. Alternatives: Kaggle (weekly GPU quota), Lambda Labs (limited free).

## 1. Clone repo and install dependencies

In [ ]:
# Clone the repo (replace REPO_URL with your fork or the project repo)
REPO_URL = "https://github.com/your-org/ocr-agentic-rag.git"  # edit if needed
!git clone $REPO_URL ocr-agentic-rag 2>/dev/null || (cd ocr-agentic-rag && git pull)
%cd ocr-agentic-rag

# Install only what's needed for embedding (lightweight)
!pip install -q sentence-transformers faiss-cpu rank_bm25 llama-index-core

## 2. Upload train_qa.json (if not in repo)

If the repo doesn't include `data/rag/FinQA/train/train_qa.json`, run the cell below and upload the file. Otherwise skip.

In [ ]:
from google.colab import files
from pathlib import Path

train_dir = Path("data/rag/FinQA/train")
train_dir.mkdir(parents=True, exist_ok=True)
if not (train_dir / "train_qa.json").exists():
    print("Upload train_qa.json (from https://github.com/czyssrs/FinQA dataset/train.json)")
    uploaded = files.upload()  # noqa: F821
    for name in uploaded:
        (train_dir / name).write_bytes(uploaded[name])
else:
    print("train_qa.json already present.")

## 3. Build index on GPU and save bundle

In [ ]:
!python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --batch_size 256

## 4. Zip and download

In [ ]:
import shutil
from pathlib import Path

src = Path("data/rag/FinQA/train/finqa_retriever_index")
zip_path = Path("finqa_retriever_index.zip")
if src.exists():
    shutil.make_archive(zip_path.with_suffix(""), "zip", src.parent, src.name)
    print(f"Created {zip_path}. Download it (right-click in file browser or use files.download).")
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Index folder not found. Run the previous cell first.")